# Evo 2 pathogenicity head — V3 fine-tune + zero-shot baseline + VCF uncertainty

A linear head sits on Evo 2 **embeddings** and outputs a **pathogenic probability**
(1 = pathogenic, 0 = benign), fused with a calibrated zero-shot **-Δlikelihood** score.

| Model | What trains | Cost |
|-------|-------------|------|
| **zero-shot -Δ** | nothing (Evo 2 likelihood ratio) | free |
| **V3** | linear head + last N Evo 2 blocks (end-to-end) | heavy (evo2_7b, cropped context) |

The old V2 frozen-embedding head was **removed** (weaker, ~0.89 test AUROC vs V3 ~0.94).

After training we compute **Variant Confidence Fingerprinting (VCF)** — per-variant
uncertainty from the model's output probability — for every binary variant, and
optionally run the trained V3 over the unlabeled **VUS** set to get pathogenicity
scores + VCF for downstream figures.

**GPU required.**


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
PROJECT_DIR = None  # e.g. "/content/drive/MyDrive/GFM_adversarial_attack"
import os
from pathlib import Path
def find_project_dir():
    if PROJECT_DIR:
        return Path(PROJECT_DIR)
    drive = Path("/content/drive")
    for c in [drive/'MyDrive'/'AD_AR_Diffusion']:
        if (c/'data').is_dir():
            return c
    raise FileNotFoundError('Set PROJECT_DIR to your Drive project folder.')
project = find_project_dir(); os.chdir(project)
print('Working directory:', project)

In [ ]:
!pip install -U accelerate
!pip install "transformers==4.57.0"
!pip install evo2
!pip uninstall flash-attn -y
!pip install flash-attn --no-build-isolation
!pip install scikit-learn peft

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 1. Config

In [ ]:
# Data (each CSV needs wt_seq, mut_seq, labels; start_pos used to center V3 crops)
DATASETS = [
    "data/ad_variants_dna_8192_binary.csv",
]
MAX_VARIANTS = None            # set e.g. 20 for a quick smoke test

# Shared split
TEST_FRAC, VAL_FRAC, SEED = 0.2, 0.2, 0

# ---- Version 3 (fine-tune: head + last N Evo2 blocks) ----
# A100 does NOT support FP8, so evo2_1b_base (needs Transformer Engine FP8) is unreliable.
# Use evo2_7b_base (no TE/FP8 required) + partial unfreeze + cropped context.
V3_MODEL = "evo2_7b_base"
V3_EMB_LAYER = "blocks.28.mlp.l3"  # 7B layer; printed on failure if wrong
V3_CONTEXT = None                  # crop centered on variant; lower to 1024 if OOM on A100
V3_FEATURE = "diff"                # 'mut' = 1 forward/variant (cheap); 'diff' = 2 forwards
V3_EPOCHS, V3_LR, V3_WD = 3, 1e-4, 1e-2
V3_ACCUM = 8                       # gradient accumulation steps (effective batch)
V3_UNFREEZE_BLOCKS = 2             # fine-tune only the last N Evo2 blocks

# ---- Fused loss ----
# total loss = BCE(embedding head) + SCORE_LOSS_WEIGHT * BCE(calibrated -Delta score);
# final probability = sigmoid(embedding logit + scalar logit)  [fused logit]
SCORE_LOSS_WEIGHT = 1.0

# ---- VUS inference (unlabeled variants of uncertain significance) ----
RUN_VUS = True
VUS_DATASET = "data/ad_variants_dna_8192_vus.csv"

import numpy as np, pandas as pd, torch
np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'


## 2. Load data + shared stratified split

In [ ]:
import re
from sklearn.metrics import roc_auc_score, average_precision_score
DNA = re.compile(r'^[ACGTN]+$', re.IGNORECASE)

def clean_variants(raw):
    """Keep rows with valid, equal-length DNA wt/mut sequences."""
    rows = []
    for _, r in raw.iterrows():
        wt, mut = str(r['wt_seq']).upper().strip(), str(r['mut_seq']).upper().strip()
        if wt and mut and DNA.match(wt) and DNA.match(mut) and len(wt) == len(mut):
            rows.append({**r.to_dict(), 'wt_seq': wt, 'mut_seq': mut})
    return pd.DataFrame(rows).reset_index(drop=True)

def zeroshot_neg_delta(model, frame):
    """Zero-shot Evo2 pathogenicity score = -(score(mut) - score(wt)), summed log-likelihood."""
    rs, ascore = [], []
    for _, r in frame.iterrows():
        rs.append(float(np.array(model.score_sequences([r['wt_seq']], reduce_method='sum'))[0]))
        ascore.append(float(np.array(model.score_sequences([r['mut_seq']], reduce_method='sum'))[0]))
    return (-(np.array(ascore) - np.array(rs))).astype(np.float32)

frames = []
for p in DATASETS:
    d = pd.read_csv(p); d['__source'] = p; frames.append(d)
raw = pd.concat(frames, ignore_index=True)
if MAX_VARIANTS is not None:
    raw = raw.head(MAX_VARIANTS)
df = clean_variants(raw)
labels = df['labels'].astype(int).to_numpy()

from sklearn.model_selection import train_test_split
idx = np.arange(len(df))
tr_idx, te_idx = train_test_split(idx, test_size=TEST_FRAC, stratify=labels, random_state=SEED)
tr_idx, va_idx = train_test_split(tr_idx, test_size=VAL_FRAC, stratify=labels[tr_idx], random_state=SEED)
ytr, yva, yte = labels[tr_idx], labels[va_idx], labels[te_idx]
print(f'{len(df)} variants | path={labels.sum()} benign={(labels==0).sum()}')
print(f'split: train={len(tr_idx)} val={len(va_idx)} test={len(te_idx)}')


## 3. Zero-shot baseline — Evo 2 -Δlikelihood

Score every variant once with Evo 2's summed log-likelihood: `neg_delta = -(score(mut) - score(wt))`.
Cached to Drive. This is the free baseline **and** the second (calibrated) loss term for V3.


In [ ]:
from evo2 import Evo2
import time

cache = Path(f"data/evo2_zeroshot_{V3_MODEL}.npz")
if cache.is_file() and len(np.load(cache)['labels']) == len(df):
    z = np.load(cache); neg_delta = z['neg_delta']
    print('Loaded cached zero-shot scores', neg_delta.shape)
else:
    m = Evo2(V3_MODEL)
    t0 = time.time()
    rs_l, as_l = [], []
    for i, r in df.iterrows():
        rs_l.append(float(np.array(m.score_sequences([r['wt_seq']], reduce_method='sum'))[0]))
        as_l.append(float(np.array(m.score_sequences([r['mut_seq']], reduce_method='sum'))[0]))
        if (i + 1) % 25 == 0:
            print(f'  {i+1}/{len(df)} ({(time.time()-t0)/(i+1):.1f}s/variant)')
    neg_delta = (-(np.array(as_l) - np.array(rs_l))).astype(np.float32)
    np.savez_compressed(cache, neg_delta=neg_delta, labels=labels)
    print('Saved', cache, neg_delta.shape)
    del m; torch.cuda.empty_cache()


In [ ]:
from sklearn.preprocessing import StandardScaler

# standardized zero-shot pathogenicity score (-Delta) -> fused loss / logit
delta_scaler = StandardScaler().fit(neg_delta[tr_idx].reshape(-1, 1))
def D(sel): return delta_scaler.transform(neg_delta[sel].reshape(-1, 1)).astype(np.float32)
nd_all = delta_scaler.transform(neg_delta.reshape(-1, 1)).astype(np.float32)
print('zero-shot scores standardized | train positive rate', round(float(ytr.mean()), 3))


## 4. Version 3 — Evo 2 + head, partial fine-tune (BCE)

End-to-end: gradients update the **last `V3_UNFREEZE_BLOCKS` Evo 2 blocks** plus the head
(simpler and more robust than LoRA, which is incompatible with Evo 2's custom config). Embeddings
are recomputed every step (no caching). Uses `evo2_7b_base`, a cropped context window, grad
accumulation, and gradient checkpointing.

> If the embedding hook fails, the cell prints valid layer names.

In [ ]:
import re
import torch.nn as nn

evo3 = Evo2(V3_MODEL)
base = evo3.model

# Evo2 loads weights as inference tensors (can't be trained). Clone every parameter/buffer
# into a normal tensor outside InferenceMode so requires_grad can be set and autograd works.
_n_conv = 0
for module in base.modules():
    for pname, p in list(module._parameters.items()):
        if p is not None:
            module._parameters[pname] = nn.Parameter(p.clone(), requires_grad=False); _n_conv += 1
    for bname, b in list(module._buffers.items()):
        if b is not None:
            module._buffers[bname] = b.clone()
torch.cuda.empty_cache()
print(f'De-inferenced {_n_conv} parameter tensors (now trainable-capable)')

for p in base.parameters(): p.requires_grad_(False)
if hasattr(base, 'gradient_checkpointing_enable'):
    try: base.gradient_checkpointing_enable()
    except Exception: pass

# unfreeze only the last V3_UNFREEZE_BLOCKS transformer/hyena blocks (by 'blocks.<idx>' name)
block_idxs = set()
for n, _ in base.named_parameters():
    m = re.search(r'blocks\.(\d+)\.', n)
    if m: block_idxs.add(int(m.group(1)))
keep = set(sorted(block_idxs)[-V3_UNFREEZE_BLOCKS:]) if block_idxs else set()
trainable_names = []
for n, p in base.named_parameters():
    m = re.search(r'blocks\.(\d+)\.', n)
    if m and int(m.group(1)) in keep:
        p.requires_grad_(True); trainable_names.append(n)
n_tr = sum(p.numel() for p in base.parameters() if p.requires_grad)
n_all = sum(p.numel() for p in base.parameters())
print(f'Unfreezing blocks {sorted(keep)} | trainable Evo2 params {n_tr:,} / {n_all:,} '
      f'({100*n_tr/max(n_all,1):.2f}%)')

# locate embedding layer module for a forward hook
emb_mod = None
for n, m in base.named_modules():
    if n.endswith(V3_EMB_LAYER):
        emb_mod = m; break
if emb_mod is None:
    cand = [n for n, m in base.named_modules() if 'mlp.l3' in n]
    raise RuntimeError(f"V3_EMB_LAYER '{V3_EMB_LAYER}' not found; candidates: {cand[:8]}")
print('Hooked embedding layer:', V3_EMB_LAYER)

In [ ]:
def crop_center(seq, center_1based, width):
    if width is None or width >= len(seq): return seq
    half = width // 2
    c = max(0, min(len(seq), center_1based - 1))
    start = max(0, c - half); end = min(len(seq), start + width)
    start = max(0, end - width)
    return seq[start:end]

_cap = {}
def _hook(mod, inp, out): _cap['h'] = out[0] if isinstance(out, tuple) else out
_handle = emb_mod.register_forward_hook(_hook)

def evo_embed_grad(seq):
    ids = torch.tensor(evo3.tokenizer.tokenize(seq), dtype=torch.int)[None].to(device)
    _cap.clear()
    try:
        evo3.model(ids)
    except TypeError:
        evo3.model(input_ids=ids)
    h = _cap['h']                     # [1, L, D]
    return h.float().mean(1).squeeze(0)  # [D]

def variant_embed(row):
    mut = crop_center(row['mut_seq'], int(row.get('start_pos', len(row['mut_seq'])//2)), V3_CONTEXT)
    if V3_FEATURE == 'diff':
        wt = crop_center(row['wt_seq'], int(row.get('start_pos', len(row['wt_seq'])//2)), V3_CONTEXT)
        return evo_embed_grad(mut) - evo_embed_grad(wt)
    return evo_embed_grad(mut)

# infer embedding dim with one no-grad pass
with torch.no_grad():
    _d = variant_embed(df.iloc[int(tr_idx[0])]).shape[0]
print('V3 embedding dim', _d)

In [ ]:
head3 = nn.Sequential(nn.Linear(_d, 1)).to(device)  # linear head on fine-tuned embeddings
scalar_head3 = nn.Linear(1, 1).to(device)            # calibrates Evo2 -Delta score (2nd loss term)
def score_tensor(j):
    return torch.tensor(nd_all[int(j)], device=device)

params = ([p for p in evo3.model.parameters() if p.requires_grad]
          + list(head3.parameters()) + list(scalar_head3.parameters()))
opt3 = torch.optim.AdamW(params, lr=V3_LR, weight_decay=V3_WD)
pw3 = torch.tensor([(ytr == 0).sum()/max((ytr == 1).sum(),1)], device=device)
bce3 = nn.BCEWithLogitsLoss(pos_weight=pw3)

def run_split(indices):
    preds = []
    for j in indices:
        e = variant_embed(df.iloc[int(j)])
        fused = head3(e).squeeze(-1) + scalar_head3(score_tensor(j)).squeeze(-1)
        preds.append(torch.sigmoid(fused).item())
    return np.array(preds)

best_auc3, best3 = -1, None
for ep in range(V3_EPOCHS):
    evo3.model.train(); head3.train(); scalar_head3.train()
    order = np.random.permutation(tr_idx)
    opt3.zero_grad()
    for step, j in enumerate(order):
        row = df.iloc[int(j)]
        e = variant_embed(row)
        logit_embed = head3(e).squeeze(-1)
        logit_scalar = scalar_head3(score_tensor(j)).squeeze(-1)
        y = torch.tensor([float(labels[int(j)])], device=device)
        loss = (bce3(logit_embed.unsqueeze(0), y)
                + SCORE_LOSS_WEIGHT * bce3(logit_scalar.unsqueeze(0), y)) / V3_ACCUM
        loss.backward()
        if (step+1) % V3_ACCUM == 0:
            opt3.step(); opt3.zero_grad()
    opt3.step(); opt3.zero_grad()
    evo3.model.eval(); head3.eval(); scalar_head3.eval()
    with torch.no_grad(): pv = run_split(va_idx)
    auc = roc_auc_score(yva, pv) if len(np.unique(yva)) > 1 else float('nan')
    if auc > best_auc3:
        best_auc3 = auc
        tr_set = set(trainable_names)
        best3 = ({k: v.detach().clone() for k, v in evo3.model.state_dict().items() if k in tr_set},
                 {k: v.detach().clone() for k, v in head3.state_dict().items()},
                 {k: v.detach().clone() for k, v in scalar_head3.state_dict().items()})
    print(f'V3 epoch {ep+1}/{V3_EPOCHS} val AUROC {auc:.4f}')

if best3 is not None:
    finetune_sd, head_sd, scalar_sd = best3
    evo3.model.load_state_dict(finetune_sd, strict=False)
    head3.load_state_dict(head_sd); scalar_head3.load_state_dict(scalar_sd)
evo3.model.eval(); head3.eval(); scalar_head3.eval()

# V3 pathogenicity probability for EVERY binary variant (needed for VCF on all splits)
with torch.no_grad():
    v3_all = run_split(np.arange(len(df)))
v3_test = v3_all[te_idx]
print(f'V3 best val AUROC {best_auc3:.4f} | test AUROC {roc_auc_score(yte, v3_test):.4f} '
      f'AUPRC {average_precision_score(yte, v3_test):.4f}')
# NOTE: hook stays registered here; the VUS cell re-uses variant_embed then removes it.


## 5. Compare V3 vs zero-shot baseline

In [ ]:
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

def boot_ci(y, p, metric=roc_auc_score, n=2000):
    rs = np.random.default_rng(SEED); vals = []
    for _ in range(n):
        s = rs.integers(0, len(y), len(y))
        if len(np.unique(y[s])) < 2: continue
        vals.append(metric(y[s], p[s]))
    return np.percentile(vals, [2.5, 97.5])

zs_test = neg_delta[te_idx]
rows_tbl = []
for name, p in [('zero-shot -d', zs_test), ('V3 finetune', v3_test)]:
    lo, hi = boot_ci(yte, p)
    rows_tbl.append({'model': name, 'AUROC': roc_auc_score(yte, p),
                     'AUROC_lo': lo, 'AUROC_hi': hi, 'AUPRC': average_precision_score(yte, p)})
print(pd.DataFrame(rows_tbl).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for name, p in [('zero-shot -d', zs_test), ('V3 finetune', v3_test)]:
    fpr, tpr, _ = roc_curve(yte, p)
    ax[0].plot(fpr, tpr, label=f'{name}, AUROC={roc_auc_score(yte, p):.3f}')
ax[0].plot([0,1],[0,1],'k--',alpha=.4); ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR')
ax[0].set_title('Test ROC'); ax[0].legend(loc='lower right')
ax[1].hist(v3_test[yte==0], bins=20, alpha=.6, label='benign')
ax[1].hist(v3_test[yte==1], bins=20, alpha=.6, label='pathogenic')
ax[1].set_title('V3 pathogenicity prob (path->1, benign->0)'); ax[1].set_xlabel('prob'); ax[1].legend()
plt.tight_layout(); plt.show()


## 6. Variant Confidence Fingerprinting (VCF)

Per-variant uncertainty read directly off the model's output probability
`p = P(pathogenic)`, with two-class distribution `(1-p, p)`:

- **uncertainty** = Shannon **entropy** = `-Σ_i p_i·log2 p_i = -p·log2 p - (1-p)·log2(1-p)`
  (bits, in [0,1]; 0 = fully confident, 1 = maximally uncertain at p=0.5)
- **confidence** = `max(p, 1-p)` (kept as a convenience column)

For a binary head both are monotone in `|p - 0.5|`, so VCF is a *calibrated
boundary-distance* signal — not epistemic (model) uncertainty. It is the classifier-side
analogue of the DYNA mixture entropy `H(s_v)` in the framework, and lets us ask whether
uncertain variants (VUS) span a wider entropy range than labeled Pathogenic/Benign.

We compute VCF for **two** probability sources, over **all** sequences (train/val/test + VUS):
- **V3**: the fine-tuned head's fused probability.
- **zero-shot**: the `-Δlikelihood` score mapped to a probability by Platt scaling
  (a 2-parameter logistic fit on the train split). Entropy needs a probability in [0,1];
  the raw `-Δ` is not one, so this single monotone 1-D calibration is the minimal step.


In [ ]:
def vcf_from_prob(p):
    """Variant Confidence Fingerprinting from P(pathogenic).
    Uncertainty = Shannon entropy -sum_i p_i log2 p_i over the two-class distribution."""
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    entropy = -(p * np.log2(p) + (1 - p) * np.log2(1 - p))   # uncertainty (bits, [0,1])
    conf = np.maximum(p, 1 - p)
    pred = (p >= 0.5).astype(int)
    return entropy, conf, pred

# zero-shot score -> probability via Platt scaling (2-param logistic, fit on TRAIN only).
# Entropy needs p in [0,1]; raw -Delta is not a probability. Applied to ALL sequences.
from sklearn.linear_model import LogisticRegression
zs_cal = LogisticRegression(max_iter=1000).fit(neg_delta[tr_idx].reshape(-1, 1), ytr)
def zeroshot_prob(nd):
    return zs_cal.predict_proba(np.asarray(nd, dtype=float).reshape(-1, 1))[:, 1]

def add_scores(frame, neg_d, v3_p):
    """Attach zero-shot + V3 pathogenicity probs and their VCF entropy to a copy of frame."""
    f = frame.copy()
    zs_p = zeroshot_prob(neg_d)
    zs_ent, zs_conf, zs_pred = vcf_from_prob(zs_p)
    v3_ent, v3_conf, v3_pred = vcf_from_prob(v3_p)
    f['zeroshot_neg_delta_likelihood'] = neg_d          # raw zero-shot score
    f['zeroshot_pathogenicity_prob']   = zs_p           # calibrated zero-shot prob
    f['zeroshot_vcf_entropy']          = zs_ent         # zero-shot uncertainty (bits)
    f['zeroshot_vcf_confidence']       = zs_conf
    f['zeroshot_pred_label']           = zs_pred
    f['v3_pathogenicity_prob']         = v3_p           # fine-tuned prob
    f['v3_vcf_entropy']                = v3_ent         # V3 uncertainty (bits)
    f['v3_vcf_confidence']             = v3_conf
    f['v3_pred_label']                 = v3_pred
    return f

split = np.array(['train'] * len(df), dtype=object)
split[va_idx] = 'val'; split[te_idx] = 'test'

out = add_scores(df, neg_delta, v3_all)   # ALL binary sequences (train/val/test)
out['split'] = split
out.to_csv('data/ad_binary_scores_v3_vcf.csv', index=False)
torch.save({'head3': head3.state_dict(), 'scalar_head3': scalar_head3.state_dict()},
           'data/v3_heads.pt')
print('Wrote data/ad_binary_scores_v3_vcf.csv', out.shape, 'and data/v3_heads.pt')
out[['zeroshot_pathogenicity_prob', 'zeroshot_vcf_entropy',
     'v3_pathogenicity_prob', 'v3_vcf_entropy']].describe()


## 7. VUS inference (variants of uncertain significance)

VUS have **no ground-truth labels**, so there is no AUROC here — this is a pure
inference pass with the trained V3. VUS are the whole point of the project: the VCF
distribution over VUS (do they cluster near the boundary? do they span a wide range?)
is the direct, testable readout of the "VUS are not uniformly uncertain" hypothesis.
Treat the pathogenicity probs as **exploratory** — V3 was trained only on labeled
Pathogenic/Benign, so applying it to VUS assumes no severe distribution shift.


In [ ]:
if RUN_VUS:
    vus_raw = pd.read_csv(VUS_DATASET)
    vus = clean_variants(vus_raw)
    print(f'VUS loaded: {len(vus_raw)} rows -> {len(vus)} valid DNA variants')

    # re-register the embedding hook removed nowhere yet; safe to (re)attach
    _handle_vus = emb_mod.register_forward_hook(_hook)
    evo3.model.eval(); head3.eval(); scalar_head3.eval()

    vus_neg_delta = zeroshot_neg_delta(evo3, vus)                       # zero-shot score
    vus_nd_std = delta_scaler.transform(vus_neg_delta.reshape(-1, 1)).astype(np.float32)

    probs = []
    with torch.no_grad():
        for k in range(len(vus)):
            row = vus.iloc[k]
            e = variant_embed(row)
            fused = head3(e).squeeze(-1) + scalar_head3(torch.tensor(vus_nd_std[k], device=device)).squeeze(-1)
            probs.append(torch.sigmoid(fused).item())
    _handle_vus.remove()
    vus_prob = np.array(probs)

    # same zero-shot + V3 columns as the binary table (zero-shot calibrator reused)
    vout = add_scores(vus, vus_neg_delta, vus_prob)
    vout['split'] = 'vus'
    vout.to_csv('data/ad_vus_scores_v3_vcf.csv', index=False)
    print('Wrote data/ad_vus_scores_v3_vcf.csv', vout.shape)

    # combined table for figures (binary + VUS); union of columns
    out['dataset'] = 'binary'
    vout['dataset'] = 'vus'
    combined = pd.concat([out, vout], ignore_index=True, sort=False)
    combined.to_csv('data/ad_all_scores_v3_vcf.csv', index=False)
    print('Wrote data/ad_all_scores_v3_vcf.csv', combined.shape)

    display(vout[['zeroshot_pathogenicity_prob', 'zeroshot_vcf_entropy',
                  'v3_pathogenicity_prob', 'v3_vcf_entropy']].describe())
else:
    print('RUN_VUS = False; skipped VUS inference.')


## 8. VCF summary by ClinVar category (figure-ready)

Groups mean VCF entropy by ClinVar category if a category column exists
(e.g. `clinvar_category`, `clinical_significance`). This is the plot behind Prediction 3
(diversity/uncertainty ordering: Conflicting > VUS > Pathogenic/Benign).

In [ ]:
cat_col = next((c for c in ['clinvar_category', 'clinical_significance', 'clnsig', 'category']
                 if 'combined' in dir() and c in combined.columns), None)
if 'combined' in dir() and cat_col:
    summ = (combined.groupby(cat_col)[['zeroshot_vcf_entropy', 'v3_vcf_entropy',
                                       'zeroshot_pathogenicity_prob', 'v3_pathogenicity_prob']]
            .agg(['mean', 'std', 'count']))
    print(f'VCF grouped by {cat_col}:')
    print(summ.to_string())
    summ.to_csv('data/ad_vcf_summary_by_category.csv')
    print('Wrote data/ad_vcf_summary_by_category.csv')
else:
    print('No ClinVar category column found; combined CSV still has per-variant VCF for plotting.')
